# Excitatory whole-neuron morphology ETL

Register VISp WNM cells, three morphology feature matrices, MET-type mappings, and ipsilateral/contralateral projections.

In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models, write_projection_matrix
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    CellToClusterMapping,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Laterality,
    MappingSet,
    Modality,
    ProjectionMeasurementMatrix,
    ProjectionMeasurementType,
    Unit,
)

In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EXC WNM"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "visp_wnm"
DATASET_ID = "visp_exc_wnm"
METTYPE_HIERARCHY_ID = "visp_met_types_taxonomy"
MAPPING_SET_ID = "visp_exc_wnm_mettype_mapping"
SHARED_FEATURE_SET_ID = "exc_visp_morph_features"
LOCAL_AXON_FEATURE_SET_ID = "wnm_exc_local_axon_features"
COMPLETE_AXON_FEATURE_SET_ID = "wnm_exc_complete_axon_features"
IPSI_MATRIX_ID = "wnm_exc_proj_ipsi"
CONTRA_MATRIX_ID = "wnm_exc_proj_contra"
PUBLICATION = "https://doi.org/10.1101/2023.11.25.568393"

METADATA_PATH = DATA_ROOT / "FullMorphMetaData_Master.csv"
SHARED_FEATURES_PATH = DATA_ROOT / "RawFeaturesWide_ChamferCorr.csv"
LOCAL_AXON_PATH = DATA_ROOT / "AxonRawReatureWide.csv"
COMPLETE_AXON_PATH = DATA_ROOT / "fMOST_Complete_Axon_Features.csv"
PROJECTION_PATH = DATA_ROOT / "ProjectionMatrix_tip_and_branch_roll_up.csv"
for path in (METADATA_PATH, SHARED_FEATURES_PATH, LOCAL_AXON_PATH, COMPLETE_AXON_PATH, PROJECTION_PATH):
    assert path.is_file(), f"Missing local input: {path}"

## Normalize source tables

In [ ]:
def normalize_cell_id(value):
    return Path(str(value)).name.removesuffix(".swc")

metadata = pd.read_csv(METADATA_PATH, index_col=0)
metadata.index = metadata.index.map(normalize_cell_id)
metadata.index.name = "id"

shared_features = pd.read_csv(SHARED_FEATURES_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
shared_features = shared_features.rename(columns={"swc_path": "id"})
shared_features["id"] = shared_features["id"].map(normalize_cell_id)

local_axon_features = pd.read_csv(LOCAL_AXON_PATH).rename(columns={"specimen_id": "id"})
local_axon_features["id"] = local_axon_features["id"].map(normalize_cell_id)

complete_axon_features = pd.read_csv(COMPLETE_AXON_PATH, index_col=0)
complete_axon_features.index = complete_axon_features.index.map(normalize_cell_id)
complete_axon_features.index.name = "id"
complete_axon_features = complete_axon_features.reset_index()

projections = pd.read_csv(PROJECTION_PATH, index_col=0)
projections.index = projections.index.map(normalize_cell_id)
projections.index.name = "id"

for frame in (metadata, shared_features, local_axon_features, complete_axon_features, projections):
    assert frame.index.is_unique if frame.index.name == "id" else frame["id"].is_unique

## Dataset and DataItems

In [ ]:
# Register every source ID before writing matrices or mappings that reference cells.
registered_ids = (
    set(metadata.index)
    | set(shared_features["id"])
    | set(local_axon_features["id"])
    | set(complete_axon_features["id"])
    | set(projections.index)
)
dataset = DataSet(
    id=DATASET_ID,
    name="VISp excitatory whole-neuron morphology cells",
    publication=PUBLICATION,
    modality=Modality.MORPHOLOGY,
    project_id=PROJECT_ID,
)
dataitems = [DataItem(id=cell_id, name=cell_id, project_id=PROJECT_ID) for cell_id in sorted(registered_ids)]
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cell_id, dataset_id=DATASET_ID, project_id=PROJECT_ID
    )
    for cell_id in sorted(registered_ids)
]
print(write_models([dataset], output_root=OUTPUT_ROOT))
print(write_models(dataitems, output_root=OUTPUT_ROOT))
print(write_models(associations, output_root=OUTPUT_ROOT))

## Morphology feature matrices

In [ ]:
def write_feature_payload(frame, feature_set_id, description, extraction_method):
    feature_columns = [column for column in frame.columns if column != "id"]
    assert feature_columns and frame["id"].is_unique
    definitions = [
        CellFeatureDefinition(
            id=column,
            description=column.replace("_", " "),
            data_type="<f8",
            project_id=PROJECT_ID,
            feature_set_id=feature_set_id,
        )
        for column in feature_columns
    ]
    feature_set = CellFeatureSet(
        id=feature_set_id,
        description=description,
        feature_definition_ids=feature_columns,
        extraction_method=extraction_method,
        project_id=PROJECT_ID,
    )
    print(write_models(definitions, output_root=OUTPUT_ROOT))
    print(write_models([feature_set], output_root=OUTPUT_ROOT))

    # A wide payload is keyed by DataItem id and scoped by project and feature set.
    wide = frame[["id", *feature_columns]].copy()
    wide[feature_columns] = wide[feature_columns].astype("float64")
    wide.insert(1, "project_id", PROJECT_ID)
    wide.insert(2, "feature_set_id", feature_set_id)
    schema = build_cell_feature_matrix_schema(feature_set, definitions, cell_index_column="id")
    table = pa.Table.from_pandas(wide, schema=schema, preserve_index=False)
    payload_path = OUTPUT_ROOT / f"cellfeatures/{feature_set_id}"
    write_deltalake(
        payload_path,
        table,
        mode="overwrite",
        predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{feature_set_id}'",
        partition_by=["project_id", "feature_set_id"],
    )
    matrix = CellFeatureMatrix(
        id=f"{PROJECT_ID}_{feature_set_id}",
        feature_set_id=feature_set_id,
        parquet_path=f"file://{payload_path.resolve()}/",
        cell_index_column="id",
        project_id=PROJECT_ID,
    )
    print(write_models([matrix], output_root=OUTPUT_ROOT))
    return payload_path, set(wide["id"]), set(feature_columns)

In [ ]:
feature_payloads = {}
feature_payloads[SHARED_FEATURE_SET_ID] = write_feature_payload(
    shared_features,
    SHARED_FEATURE_SET_ID,
    "Dendrite morphology features shared with the excitatory Patch-seq feature vocabulary.",
    "Whole-neuron morphology reconstruction feature extraction",
)
feature_payloads[LOCAL_AXON_FEATURE_SET_ID] = write_feature_payload(
    local_axon_features,
    LOCAL_AXON_FEATURE_SET_ID,
    "Local axon and apical dendrite morphology features for WNM cells.",
    "Local axon morphology feature extraction",
)
feature_payloads[COMPLETE_AXON_FEATURE_SET_ID] = write_feature_payload(
    complete_axon_features,
    COMPLETE_AXON_FEATURE_SET_ID,
    "Complete-axon morphology features from fMOST reconstructions.",
    "Complete-axon fMOST feature extraction",
)

## MET-type mappings

In [ ]:
mettype_clusters = pl.read_delta(OUTPUT_ROOT / "cluster").filter(
    pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID
)
assert not mettype_clusters.is_empty(), "Run the shared MET-type taxonomy ETL first."
mettype_parent = dict(mettype_clusters.select("id", "parent").iter_rows())
mapped_cells = metadata.dropna(subset=["predicted_met_type"]).copy()
unknown_mettypes = set(mapped_cells["predicted_met_type"]) - set(mettype_parent)
assert not unknown_mettypes, f"Unknown MET-types: {sorted(unknown_mettypes)}"
mapping_set = MappingSet(
    id=MAPPING_SET_ID,
    name="VISp WNM excitatory MET-type assignments",
    description=(
        "Routed random forest mapping of WNM cells onto the VISp MET taxonomy; "
        "the classifier was trained on the excitatory Patch-seq mMET cohort."
    ),
    method_name="Routed random forest mapping",
    source_dataset=DATASET_ID,
    target_hierarchy=METTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
mappings = []
for cell_id, row in mapped_cells.iterrows():
    leaf_probability = float(row["probability"]) if pd.notna(row["probability"]) else None
    for cluster_id, is_leaf in walk_ancestors(row["predicted_met_type"], mettype_parent):
        mappings.append(
            CellToClusterMapping(
                id=f"{cell_id}-{cluster_id}-{PROJECT_ID}-{METTYPE_HIERARCHY_ID}",
                mapping_set=MAPPING_SET_ID,
                source_cell=str(cell_id),
                target_cluster=cluster_id,
                probability=leaf_probability if is_leaf else None,
                project_id=PROJECT_ID,
            )
        )
print(write_models([mapping_set], output_root=OUTPUT_ROOT))
print(write_models(mappings, output_root=OUTPUT_ROOT))

## Projection matrices

Region indexes currently use source CCF acronyms, and `values` points to a file-backed Delta payload while the schema evolves toward registered BrainRegion and Zarr references. `MICRONS_OF_AXON` follows the source value scale and should be confirmed with the data owner.

In [ ]:
ipsi_columns = [column for column in projections.columns if column.startswith("ipsi_")]
contra_columns = [column for column in projections.columns if column.startswith("contra_")]
other_columns = [column for column in projections.columns if column not in {*ipsi_columns, *contra_columns}]
assert not other_columns
ipsi_regions = [column.removeprefix("ipsi_") for column in ipsi_columns]
contra_regions = [column.removeprefix("contra_") for column in contra_columns]
assert len(ipsi_regions) == len(set(ipsi_regions))
assert len(contra_regions) == len(set(contra_regions))
projection_ids = list(projections.index)

In [ ]:
def write_projection_payload(matrix_id, source_columns, regions, laterality, description):
    wide = projections[source_columns].copy()
    wide.columns = regions
    wide = wide.reset_index()
    wide.insert(1, "project_id", PROJECT_ID)
    wide.insert(2, "dataset_id", DATASET_ID)
    schema = pa.schema(
        [
            pa.field("id", pa.string(), nullable=False),
            pa.field("project_id", pa.string(), nullable=False),
            pa.field("dataset_id", pa.string(), nullable=False),
            *[pa.field(region, pa.float64()) for region in regions],
        ]
    )
    table = pa.Table.from_pandas(wide, schema=schema, preserve_index=False)
    payload_path = OUTPUT_ROOT / f"projectionmeasurementmatrix/{matrix_id}"
    write_deltalake(
        payload_path,
        table,
        mode="overwrite",
        predicate=f"project_id = '{PROJECT_ID}'",
        partition_by=["project_id"],
    )
    matrix = ProjectionMeasurementMatrix(
        id=matrix_id,
        description=description,
        measurement_type=ProjectionMeasurementType.MICRONS_OF_AXON,
        modality=Modality.MORPHOLOGY,
        laterality=laterality,
        region_index=regions,
        data_item_index=projection_ids,
        values=f"file://{payload_path.resolve()}/",
        unit=Unit.MICRONS_LENGTH,
        project_id=PROJECT_ID,
    )
    values = projections[source_columns].to_numpy(dtype="float64")
    print(write_projection_matrix(matrix, values, output_root=OUTPUT_ROOT))
    return payload_path, values.shape

In [ ]:
ipsi_path, ipsi_shape = write_projection_payload(
    IPSI_MATRIX_ID, ipsi_columns, ipsi_regions, Laterality.IPSILATERAL,
    "WNM excitatory ipsilateral per-cell axon projection by CCF region.",
)
contra_path, contra_shape = write_projection_payload(
    CONTRA_MATRIX_ID, contra_columns, contra_regions, Laterality.CONTRALATERAL,
    "WNM excitatory contralateral per-cell axon projection by CCF region.",
)

## Verify

In [ ]:
# Verify references and matrix row sets after every scoped write is complete.
stored_associations = pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
)
stored_mappings = pl.read_delta(OUTPUT_ROOT / "celltoclustermapping").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_ID)
)
stored_projection_metadata = pl.read_delta(OUTPUT_ROOT / "projectionmeasurementmatrix").filter(
    (pl.col("project_id") == PROJECT_ID)
    & pl.col("id").is_in([IPSI_MATRIX_ID, CONTRA_MATRIX_ID])
)
assert set(stored_associations["dataitem_id"]) == registered_ids
assert stored_mappings.height == len(mappings)
assert set(stored_mappings["target_cluster"]) <= set(mettype_parent)
assert stored_mappings.filter(pl.col("probability").is_not_null()).height == len(mapped_cells)
for feature_set_id, (payload_path, expected_ids, expected_columns) in feature_payloads.items():
    stored = pl.read_delta(payload_path).filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("feature_set_id") == feature_set_id)
    )
    assert set(stored["id"]) == expected_ids
    assert set(stored.columns) == {"id", "project_id", "feature_set_id", *expected_columns}
assert stored_projection_metadata.height == 2
assert pl.read_delta(ipsi_path).height == ipsi_shape[0]
assert pl.read_delta(contra_path).height == contra_shape[0]
assert set(projection_ids) <= registered_ids
print("WNM CCM records verified.")